In [ ]:
## login wandb
import wandb
wandb.login()
## set up project name
import os
os.environ["WANDB_PROJECT"] = "chess-llm" 
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

In [ ]:
import unsloth
import vllm
import torch
import trl

print(vllm.__version__)
print(unsloth.__version__)
print(torch.__version__)
print(trl.__version__)

## Model

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B-Thinking-2507", 
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

In [ ]:
NEW_TOKENS = [  
    "♔","♕","♖","♗","♘","♙",  
    "♚","♛","♜","♝","♞","♟",
    "<uci_move>", "</uci_move>",
]
xs = tokenizer("♔♕♖♗♘♙♚♛♜♝♞♟ <uci_move>a1a2</uci_move>")
print([tokenizer.decode(x) for x in xs["input_ids"]])

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

## Data

In [ ]:
from datasets import load_dataset, Dataset
from tqdm import tqdm
import chess
import pandas as pd

In [ ]:
SYSTEM_PROMPT = """You are an expert chess player.

## Board Representation  
White King (♔) / Black King (♚): Move 1 square in any direction.
White Queen (♕) / Black Queen (♛): Move any number of squares in any direction.
White Rook (♖) / Black Rook (♜): Move any number of squares horizontally or vertically.
White Bishop (♗) / Black Bishop (♝): Move any number of squares diagonally.
White Knight (♘) / Black Knight (♞): Move in an L-shape (2 + 1); can jump pieces.
White Pawn (♙) / Black Pawn (♟): Move forward 1 (or 2 on first move); capture diagonally.

## Output Requirements  
Follow the steps below in order:  
1. Game Phase
   - Identify the current game phase (opening, middlegame, or endgame) in one sentence.  
2. Move Analysis 
   - Analyze 2–3 candidate moves in concise 2-3 sentence total.  
   - Begin with "As White/Black, ..." depending on the side to move.  
3. Final Decision
   - Select the move abd ensure it is legal in one sentence.
   - Justify your move in one sentence.
4. Move Formatting  
   - Output the chosen move in UCI format, enclosed exactly as <uci_move>your_move</uci_move>

## Note
- The move **must appear** in the provided legal moves list.
- Your output is 120 words maximum.

## Context
Your side: {side_to_move}
Legal moves:
{legal_moves_uci_list}
Board position:
{board_utf}
"""

In [ ]:
def format_prompt(row):  
    prompt = SYSTEM_PROMPT.format(
        side_to_move=row["side_to_move"],
        legal_moves_uci_list=row["legal_moves_uci_list"],
        board_utf=row["board_utf"],
    ) 
    response = f"<think>\n{row["explanation"]}\n</think>\n<uci_move>{row["target_move"]}</uci_move>"   
    return [  
        {"role": "user", "content": prompt},  
        {"role": "assistant", "content": response},  
    ]

In [ ]:
df = pd.read_parquet("../data/reasoning-exp02.parquet")
df = df[:10]
df["explanation"] = df["explanation"].apply(lambda x: x.replace("\n\n", "\n"))

## preprocess
df["prompt"] = df.apply(format_prompt, axis=1)

In [ ]:
df["text"] = tokenizer.apply_chat_template(df["prompt"].values.tolist(), tokenize=False)

In [ ]:
from sklearn.model_selection import train_test_split  
from datasets import Dataset  
  
# Unique boards  
unique_boards = df["board_utf"].unique()  
  
# Split boards, NOT rows  
train_boards, test_boards = train_test_split(  
    unique_boards,  
    test_size=0.1,  
    random_state=42,  
    shuffle=True,  
)  
  
# Filter rows  
train_df = df[df["board_utf"].isin(train_boards)].reset_index(drop=True)  
test_df  = df[df["board_utf"].isin(test_boards)].reset_index(drop=True)  
  
# Create HF datasets  
ds = {  
    "train": Dataset.from_pandas(train_df),  
    "test": Dataset.from_pandas(test_df),  
}  

In [ ]:
text = ds["train"]["text"][0]

print(text)
print(len(tokenizer(text)["input_ids"]))

## SFT

In [ ]:
import re  
import numpy as np  
  
# --------------------------------------------------  
# Logits preprocessing  
# --------------------------------------------------  
def preprocess_logits_for_metrics(logits, labels):  
    if isinstance(logits, tuple):  
        logits = logits[0]  
    return logits.argmax(dim=-1)  
  
  
# --------------------------------------------------  
# Metric computation: exact UCI match  
# --------------------------------------------------  
UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
  
def extract_uci(text):  
    match = UCI_PATTERN.search(text)  
    return match.group(1).strip() if match else None  
  
  
def make_compute_metrics(tokenizer, eval_dataset):  
  
    def compute_metrics(eval_preds):  
        preds, _ = eval_preds  
  
        if isinstance(preds, tuple):  
            preds = preds[0]  
  
        # Replace -100 so decoding works  
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)  
  
        # Decode model outputs  
        decoded_preds = tokenizer.batch_decode(  
            preds, skip_special_tokens=True  
        )  
  
        correct = 0
        missing = 0
        legal = 0
        total = len(decoded_preds)  
  
        for pred_text, example in zip(decoded_preds, eval_dataset):
            pred_move = extract_uci(pred_text)  
            target_move = example["target_move"]
            legal_moves_uci_list = example["legal_moves_uci_list"]
            print(pred_move in legal_moves_uci_list)
            print(pred_move)
            print(legal_moves_uci_list)
            print("*"*60)

            if pred_move is None:  
                missing += 1
            if pred_move == target_move:  
                correct += 1
            if pred_move in legal_moves_uci_list:
                legal += 1
  
        return {  
            "move_accuracy": round(correct / total, 4),
            "missing_tag_rate": round(missing / total, 4),
            "legal_move_rate": round(legal / total, 4),
        }  
  
    return compute_metrics  

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset= ds["train"],
    eval_dataset= ds["test"],
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,  
    compute_metrics=make_compute_metrics(tokenizer, ds["test"]), 
    args = SFTConfig(
        dataset_text_field = "text",
        optim = "adamw_8bit",
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "wandb", # Use TrackIO/WandB etc
        
        # training params
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=1,
        num_train_epochs=5,
        fp16=True,
        bf16=False,
        weight_decay = 0.001,
        
        # logging
        # eval_strategy="epoch",
        # save_strategy="epoch",
        # logging_strategy="epoch",
        eval_strategy="steps",
        save_strategy="steps",
        logging_strategy="steps",
        logging_steps=1,
        save_steps=1,
        eval_steps=1,
        save_total_limit=1,
        
        # ✅ BEST MODEL LOGIC  
        load_best_model_at_end=True,  
        metric_for_best_model="move_accuracy",  
        greater_is_better=True,  
    ),
)

In [ ]:
trainer.train()

## Test

In [ ]:
text = tokenizer.apply_chat_template(
    ds["test"][5]["prompt"][:1],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

eos_token = "<|im_end|>"
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
    eos_token_id=tokenizer.convert_tokens_to_ids(eos_token),
)

## Save

In [ ]:
model.push_to_hub_merged(
    "Norrawee/Qwen3-4B-Thinking-2507-exp02", 
    tokenizer,
    save_method = "merged_16bit", 
)

In [ ]:
wandb.finish()